In [1]:
import pandas as pd
import sqlite3

# Load your data (upload sales_data_sample.csv first using the folder icon on the left)
df = pd.read_csv("sales_data_sample.csv", encoding="latin1")

# Create a SQLite database and load the data into a table
conn = sqlite3.connect("sales.db")
df.to_sql("sales", conn, if_exists="replace", index=False)

print("Database created with", len(df), "rows")


Database created with 2823 rows


In [2]:
%load_ext sql
%sql sqlite:///sales.db

In [5]:
import pandas as pd

# The 'conn' object is available from the previous cell (TLiBm667EEg3)
query = """
SELECT COUNT(*) AS total_orders, ROUND(SUM(SALES), 2) AS total_revenue
FROM sales;
"""

result_df = pd.read_sql(query, conn)
print(result_df)

   total_orders  total_revenue
0          2823    10032628.85


In [6]:
query = """
SELECT PRODUCTLINE, ROUND(SUM(SALES), 2) AS total_revenue
FROM sales
GROUP BY PRODUCTLINE
ORDER BY total_revenue DESC;
"""
print(pd.read_sql(query, conn))

        PRODUCTLINE  total_revenue
0      Classic Cars     3919615.66
1      Vintage Cars     1903150.84
2       Motorcycles     1166388.34
3  Trucks and Buses     1127789.84
4            Planes      975003.57
5             Ships      714437.13
6            Trains      226243.47


In [7]:
query = """
SELECT CUSTOMERNAME, ROUND(SUM(SALES), 2) AS total_spent
FROM sales
GROUP BY CUSTOMERNAME
ORDER BY total_spent DESC
LIMIT 5;
"""
print(pd.read_sql(query, conn))

                   CUSTOMERNAME  total_spent
0         Euro Shopping Channel    912294.11
1  Mini Gifts Distributors Ltd.    654858.06
2    Australian Collectors, Co.    200995.41
3            Muscle Machine Inc    197736.94
4             La Rochelle Gifts    180124.90


In [8]:
query = """
SELECT ORDERNUMBER, CUSTOMERNAME, SALES
FROM sales
WHERE SALES > (SELECT AVG(SALES) FROM sales)
ORDER BY SALES DESC
LIMIT 10;
"""
print(pd.read_sql(query, conn))

   ORDERNUMBER                  CUSTOMERNAME    SALES
0        10407     The Sharp Gifts Warehouse  14082.8
1        10322  Online Diecast Creations Co.  12536.5
2        10424         Euro Shopping Channel  12001.0
3        10412         Euro Shopping Channel  11887.8
4        10403         UK Collectables, Ltd.  11886.6
5        10405                   Mini Caravy  11739.7
6        10312  Mini Gifts Distributors Ltd.  11623.7
7        10333               Mini Wheels Co.  11336.7
8        10127            Muscle Machine Inc  11279.2
9        10150       Dragon Souveniers, Ltd.  10993.5


In [9]:
query = """
SELECT
    CASE
        WHEN SALES < 2000 THEN 'Low'
        WHEN SALES BETWEEN 2000 AND 5000 THEN 'Medium'
        ELSE 'High'
    END AS sales_tier,
    COUNT(*) AS num_orders,
    ROUND(SUM(SALES), 2) AS total_revenue
FROM sales
GROUP BY sales_tier
ORDER BY total_revenue DESC;
"""
print(pd.read_sql(query, conn))


  sales_tier  num_orders  total_revenue
0     Medium        1709     5597948.87
1       High         549     3581083.00
2        Low         565      853596.98


In [10]:
query = """
SELECT COUNTRY, COUNT(*) AS num_orders, ROUND(SUM(SALES),2) AS total_revenue
FROM sales
GROUP BY COUNTRY
HAVING COUNT(*) > 100
ORDER BY total_revenue DESC;
"""
print(pd.read_sql(query, conn))

     COUNTRY  num_orders  total_revenue
0        USA        1004     3627982.83
1      Spain         342     1215686.92
2     France         314     1110916.52
3  Australia         185      630623.10
4         UK         144      478880.46
5      Italy         113      374674.31


In [11]:
query = """
SELECT
    YEAR_ID,
    ROUND(SUM(SALES), 2) AS yearly_revenue,
    ROUND(SUM(SUM(SALES)) OVER (ORDER BY YEAR_ID), 2) AS running_total
FROM sales
GROUP BY YEAR_ID
ORDER BY YEAR_ID;
"""
print(pd.read_sql(query, conn))

   YEAR_ID  yearly_revenue  running_total
0     2003      3516979.54     3516979.54
1     2004      4724162.60     8241142.14
2     2005      1791486.71    10032628.85
